## 정확한 정확하 답변을 생성했는가? - 사내 문서 Q&A 에이전트 평가 (In-house Agent Evaluation)
* LangSmith를 사용하여 체계적인 평가를 수행
* 평가 프로세스
```
Golden Dataset (LangSmith)
        │
        ▼
┌─────────────────────────────┐
│   run_agent_to_completion   │  ← 에이전트 실행
└─────────────────────────────┘
        │
        ▼
┌─────────────────────────────┐
│      Evaluators 실행         │
│  • 답변 정확성 (LLM Judge)     │
└─────────────────────────────┘
        │
        ▼
    평가 결과 (LangSmith UI)
```


|평가 지표|설명|
|------|----|
|is_final_answer_accurate|	LLM-as-Judge 방식으로 답변의 정확성 평가
|check_source_file|	에이전트가 올바른 문서를 선택했는지 확인


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langsmith import Client

# LangSmith 클라이언트 - 데이터셋 관리, 실험 실행, 결과 조회에 사용
ls_client = Client()

In [3]:
dataset = ls_client.read_dataset(dataset_name="agent-eval-sampled")

##  Grader 설정: 5가지 기준 x 1-5점 스케일

* 기존의 방식과 다른점
    * 각 항목별로의 점수를 가중평균으로 산출해서 최종 점수를 도출함
    * weighted_score = (correctness * 0.3) + (source_retrieval * 0.15) + (groundedness * 0.3) 
               + (completeness * 0.15) + (relevance_conciseness * 0.1)
    * Correctness(정확성)와 Groundedness(근거성)에 가장 높은 가중치(각 0.3)가 부여함
        * 이유: 부정확하거나 환각이 포함된 답변은 다른 기준을 아무리 잘 충족해도 근본적으로 사용할 수 없기 때문

In [4]:
grader_instructions = """You are an expert evaluator assessing the quality of responses from an AI questionnaire agent. The agent answers employee/user questions by retrieving information from internal company documents.

You will be given:
- The **user question**
- The **agent's response**
- The **retrieval results** (e.g. which documents were retrieved and their relevance scores)
- The **ground truth answer**

Evaluate the agent's response on the following 5 criteria, each scored from 1 (worst) to 5 (best).

---

### 1. Correctness (1-5)
Is the answer factually accurate?

| Score | Description |
|-------|-------------|
| 1 | Completely wrong or contradicts the ground truth |
| 2 | Mostly incorrect with minor correct elements |
| 3 | Partially correct — key facts are right but some errors present |
| 4 | Mostly correct with only minor inaccuracies |
| 5 | Fully correct and consistent with the ground truth |

### 2. Source Retrieval (1-5)
Did the agent retrieve and reference the intended/correct document(s)?

| Score | Description |
|-------|-------------|
| 1 | Retrieved completely wrong or no documents |
| 2 | Retrieved mostly irrelevant documents with only tangential relevance |
| 3 | Retrieved a mix — some relevant, some irrelevant sources |
| 4 | Retrieved the correct document(s) but also included unnecessary ones |
| 5 | Retrieved exactly the right document(s) with no irrelevant sources |

### 3. Groundedness (1-5)
Is the answer fully supported by the retrieved document content? Does it avoid hallucination?

| Score | Description |
|-------|-------------|
| 1 | Answer is entirely fabricated or unsupported by the documents |
| 2 | Most of the answer is not grounded in the documents |
| 3 | Partially grounded — some claims are supported, others are hallucinated |
| 4 | Mostly grounded with only minor unsupported details |
| 5 | Every claim in the answer is directly supported by the retrieved documents |

### 4. Completeness (1-5)
Does the response fully address all parts of the user's question?

| Score | Description |
|-------|-------------|
| 1 | Fails to address the question at all |
| 2 | Addresses only a small part of the question |
| 3 | Addresses the main point but misses important sub-questions or details |
| 4 | Covers most aspects with only minor omissions |
| 5 | Thoroughly addresses every part of the question |

### 5. Relevance & Conciseness (1-5)
Is the response focused on the question without unnecessary or off-topic information?

| Score | Description |
|-------|-------------|
| 1 | Entirely off-topic or overwhelmed with irrelevant information |
| 2 | Mostly irrelevant with excessive tangential content |
| 3 | Relevant but includes noticeable amount of unnecessary information |
| 4 | Mostly focused with only minor irrelevant details |
| 5 | Perfectly focused — every sentence directly serves the user's question |

## Weighted Overall Score

After scoring each criterion, compute a weighted overall score using the following formula:

weighted_score = (correctness * 0.3) + (source_retrieval * 0.15) + (groundedness * 0.3) + (completeness * 0.15) + (relevance_conciseness * 0.1)

This means correctness and groundedness are the most important criteria, as a response that is inaccurate or hallucinated is fundamentally unusable regardless of how well it performs on other dimensions.
"""

In [5]:
from pydantic import BaseModel, Field

class Grade(BaseModel):
    """LLM Judge가 반환하는 채점 결과 - 0~5점 스케일"""
    score: float = Field(description="The overall grade for the agent's response, on a scale from 0 to 5.")
    reasoning: str = Field(description="A step-by-step explanation of the grading decision.")

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
        model="gemini-3.1-pro-preview",
        temperature=1
    )
grader_llm = llm.with_structured_output(Grade)

## Agent 실행함수 정의
* run_agent_to_completion >> LangSmith의 evaluate()가 각 테스트 케이스마다 호출하는 함수. Golden Dataset의 질문을 받아 에이전트를 실행하고, 결과를 반환함

In [7]:
from langchain_core.messages import HumanMessage
from inhouse_agent import agent  # 사내 문서 Q&A 에이전트 (agents/ 디렉토리에 정의)

def run_agent_to_completion(inputs):
    """LangSmith evaluate()가 호출하는 에이전트 실행 함수.
    
    Args:
        inputs: Golden Dataset의 한 행 (예: {"question": "연차 휴가는 며칠?"})
    
    Returns:
        에이전트 실행 결과 (messages 리스트 포함)
    """
    question = inputs["question"]
    
    result = agent.invoke({
        "messages": [HumanMessage(content=question)]
    })

    return result

### Evaluator 함수 정의
* Evaluator : 에이전트의 출력을 정답과 비교하여 점수를 매기는 함수
* LangSmith의 evaluate()에 전달되면, 각 테스트 케이스마다 자동으로 호출
* 에이전트의 응답뿐만 아니라 검색된 문서(outputs["messages"][-2])도 함께 LLM Judge에 전달. 이를 통해 Source Retrieval, Groundedness 등 문서 기반 평가가 가능해진다.

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage

def is_final_answer_accurate(inputs, outputs, reference_outputs):
    """가중 점수 기반 evaluator (0-5점).
    
    동일한 5가지 기준을 사용하되,
    grader_instructions에 가중치 공식이 포함되어
    LLM Judge가 가중 평균을 계산하여 최종 점수를 반환
    """
    question = inputs["question"]
    answer = outputs["messages"][-1].content  # 에이전트의 최종 답변
    retrieved_docs = outputs["messages"][-2].content  # 검색된 문서
    ground_truth_response = reference_outputs["answer"]
    
    grading_input = f"""QUESTION: {question}
ANSWER: {answer}
RETRIEVED DOCUMENTS: {retrieved_docs}
GROUND TRUTH: {ground_truth_response}"""
    
    grade = grader_llm.invoke([
        SystemMessage(content=grader_instructions),
        HumanMessage(content=grading_input)
    ])

    return grade.score  # 가중 점수 반환

In [9]:
def check_source_file(inputs, outputs, reference_outputs, example):
    """에이전트가 올바른 소스 문서를 선택했는지 확인하는 evaluator.
    
    이 evaluator는 LLM을 사용하지 않고, 에이전트의 도구 호출 기록을
    직접 분석하여 프로그래밍적으로 평가합니다.
    
    Args:
        example: LangSmith Example 객체 - metadata.source에 정답 문서명이 기록됨
    
    Returns:
        int: 1(올바른 문서 선택) 또는 0(잘못된 문서 선택)
    """
    # Golden Dataset에 기록된 정답 소스 문서 (metadata에 저장)
    ground_truth_docs = example.metadata["source"]
    messages = outputs.get('messages', [])
    actual_source = None

    # 에이전트의 메시지 히스토리를 순회하며 사용된 문서를 추출
    for msg in messages:
        if not hasattr(msg, 'name') or msg.name is None:
            continue  # 도구 호출이 아닌 메시지는 건너뜀

        if msg.name == 'check_faq':
            # 1단계: FAQ 도구 결과 확인
            content = msg.content
            if "'is_in_faq': True" in content or '"is_in_faq": true' in content:
                actual_source = 'employee_benefits_and_welfare_faq'
                break  # FAQ에서 찾았으므로 더 볼 필요 없음
            # is_in_faq=False이면 continue해서 get_document_name을 찾음

        elif msg.name == 'get_document_name':
            # 2단계: FAQ가 아닌 경우, 일반 문서 검색 도구 결과 확인
            actual_source = msg.content
            break

    # 에이전트가 선택한 문서가 정답 문서 목록에 포함되는지 확인
    return int(actual_source in ground_truth_docs)

In [10]:
experiment_result = ls_client.evaluate(
    run_agent_to_completion,
    data="agent-eval-sampled",
    evaluators=[is_final_answer_accurate,check_source_file],  # 5-criteria evaluator (0-5점)
    max_concurrency=2,
    num_repetitions=1
)

/Users/a202304035/LLM_Eval_study/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'tart-channel-99' at:
https://smith.langchain.com/o/2f003b67-27f4-48d0-8856-e1db2397e3d4/datasets/e6169a73-73fd-4f8e-ab1b-cb8e5dad6485/compare?selectedSessions=4613e8c1-e62f-4341-9e4f-ca765f609a08




21it [03:41, 10.54s/it]
